# Distilling Step-by-Step: Run Experiment\n
\n
This notebook replicates the functionality of `run.py` from the [distilling-step-by-step](https://github.com/google-research/distilling-step-by-step) repository.\n
It allows running training and evaluation experiments within a Colab environment.

In [ ]:
# 1. Clone Repository & Install Dependencies\n
! git clone "https://github.com/google-research/distilling-step-by-step.git"
%cd distilling-step-by-step

# Install required packages\n
! pip install --upgrade datasets transformers[torch] evaluate nltk rouge_score absl-py

In [ ]:
# 2. Imports\n
import argparse
import os
import logging
import sys
import torch

from datasets import DatasetDict, concatenate_datasets
from transformers import AutoTokenizer

# Import from cloned repo files\n
from data_utils import CQADatasetLoader, SVAMPDatasetLoader, ESNLIDatasetLoader, ANLI1DatasetLoader, ASDivDatasetLoader
from metrics import compute_text_acc, compute_equation_acc, compute_metrics_text, compute_metrics_equation, compute_metrics_text_aux, compute_metrics_equation_aux
from train_utils import train_and_evaluate # train_utils also imports necessary transformer classes\n

# Imports potentially needed by train_utils or other parts\n
import transformers
from transformers import set_seed
from transformers.trainer_utils import get_last_checkpoint

In [ ]:
# 3. Setup Arguments, Logging, and Seed\n

# Replicate argparse arguments as a class\n
class Args:
    # --- Core Experiment Settings --- \n
    dataset = 'cqa'  # Options: 'cqa', 'svamp', 'esnli', 'anli1', 'asdiv'\n
    model_type = 'task_prefix' # Options: 'task_prefix', 'standard'\n
    from_pretrained = 'google/t5-v1_1-base'
    label_type = 'gt' # Options: 'gt', 'llm' (use ground truth or LLM labels for training)\n
    llm = 'palm' # Options: 'palm', 'gpt', None (which LLM's rationales to use, if any)\n
    subsample = 1.0 # Fraction of training data to use (e.g., 0.1 for 10%)\n
    run = 0 # Seed offset for multiple runs\n
    
    # --- Training Hyperparameters --- \n
    alpha = 0.5 # Weight for the auxiliary rationale loss (only for 'task_prefix' model_type)\n
    max_steps = 10000 # Max training steps\n
    batch_size = 32 # Total batch size (adjust based on Colab GPU memory, e.g., 16, 32, 64)\n
    grad_steps = 2 # Gradient accumulation steps (effective_batch_size = batch_size * grad_steps)\n
    lr = 5e-5 # Learning rate\n
    optimizer_name = 'AdamW' # Optimizer\n
    bf16 = False # Use bfloat16 precision (requires Ampere GPU or newer)\n
    
    # --- Evaluation and Logging --- \n
    eval_steps = 250 # Evaluate every N steps\n
    gen_max_len = 64 # Max length for generation during evaluation\n
    output_rationale = False # Whether to output rationale during evaluation (for 'task_prefix')\n
    no_log = False # Disable logging to W&B (if setup)\n
    
    # --- Input/Output & Environment --- \n
    max_input_length = 1024 # Max input sequence length\n
    local_rank = -1 # Distributed training rank (usually -1 for single GPU/CPU)\n
    parallelize = False # Use basic model parallelism (requires multiple GPUs)\n
    
    # --- Derived/TrainingArguments-related --- \n
    # These are often passed to Hugging Face's Seq2SeqTrainingArguments\n
    # Construct output_dir dynamically\n
    _llm_suffix = f'_{llm}' if llm else ''
    output_dir = f'./outputs/{dataset}_{model_type}_{label_type}{_llm_suffix}_{subsample}_{alpha}_{run}'
    seed = 42 + run
    logging_steps = 50
    save_steps = 500
    save_total_limit = 1
    predict_with_generate = True
    generation_num_beams = 1
    learning_rate = lr # Pass lr to TrainingArguments\n
    # Calculate per_device batch size based on total batch_size and available GPUs\n
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    per_device_train_batch_size = max(1, batch_size // num_gpus)
    per_device_eval_batch_size = max(1, batch_size // num_gpus)
    gradient_accumulation_steps = grad_steps
    evaluation_strategy = 'steps' # Evaluate at eval_steps intervals\n
    # bf16 needs to be set directly in TrainingArguments\n
    # bf16 = bf16 # Already defined above\n

args = Args()

# --- Setup Logging --- \n
logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO, # Set level here\n
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    datefmt='%m/%d/%Y %H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)],
)
log_level = logging.INFO # Keep variable for potential later use\n
logger.setLevel(log_level)
transformers.utils.logging.set_verbosity(log_level)
transformers.utils.logging.enable_default_handler()
transformers.utils.logging.enable_explicit_format()

logger.info(f'Args: {vars(args)}') # Log the arguments\n

# --- Set Seed --- \n
set_seed(args.seed)

# --- Create Output Directory --- \n
os.makedirs(args.output_dir, exist_ok=True)

In [ ]:
# 4. Load and Prepare Dataset

logger.info(f'Loading dataset: {args.dataset}')

#### Initialize Dataset Loader
if args.dataset == 'cqa':
    dataset_loader = CQADatasetLoader()
elif args.dataset == 'svamp':
    dataset_loader = SVAMPDatasetLoader()
elif args.dataset == 'esnli':
    dataset_loader = ESNLIDatasetLoader()
elif args.dataset == 'anli1':
    dataset_loader = ANLI1DatasetLoader()
elif args.dataset == 'asdiv':  # NOTE: for augmenting SVAMP only
    # Special case: Load both SVAMP and ASDiv
    dataset_loader = SVAMPDatasetLoader() # Base loader for methods
    dataset_loader_svamp = SVAMPDatasetLoader()
    dataset_loader_asdiv = ASDivDatasetLoader()
else:
    raise ValueError(f"Unknown dataset: {args.dataset}")

#### Load Raw Data
if args.dataset == 'asdiv':
    datasets_svamp = dataset_loader_svamp.load_from_json()
    datasets_asdiv = dataset_loader_asdiv.load_from_json()
    # Combine training sets, use SVAMP test set
    datasets = DatasetDict({
        'train': concatenate_datasets([datasets_svamp['train'], datasets_asdiv['train']]),
        'test': datasets_svamp['test']
    })
    logger.info(f"Combined SVAMP and ASDiv for training. Train size: {len(datasets['train'])}, Test size: {len(datasets['test'])}")
else:
    datasets = dataset_loader.load_from_json()
    logger.info(f"Loaded dataset splits: {list(datasets.keys())}")

#### Load LLM Predictions (Rationales and Labels) if specified
if args.llm is not None:
    logger.info(f'Loading LLM predictions ({args.llm})')
    if args.llm not in ['palm', 'gpt']:
        raise ValueError(f"Unsupported LLM type: {args.llm}")

    # Determine load function based on llm type
    load_pred_fn = dataset_loader.load_llm_preds if args.llm == 'palm' else dataset_loader.load_gpt_preds

    if args.dataset == 'asdiv':
        # Load SVAMP train preds
        train_llm_rationales_svamp, train_llm_labels_svamp = dataset_loader_svamp.load_llm_preds(split='train') if args.llm == 'palm' else dataset_loader_svamp.load_gpt_preds(split='train')
        # Load ASDiv train preds
        train_llm_rationales_asdiv, train_llm_labels_asdiv = dataset_loader_asdiv.load_llm_preds(split='train') if args.llm == 'palm' else dataset_loader_asdiv.load_gpt_preds(split='train')
        # Concatenate train preds
        train_llm_rationales = train_llm_rationales_svamp + train_llm_rationales_asdiv
        train_llm_labels = train_llm_labels_svamp + train_llm_labels_asdiv
        # Load SVAMP test preds (since test set is SVAMP)
        test_llm_rationales, test_llm_labels = dataset_loader_svamp.load_llm_preds(split='test') if args.llm == 'palm' else dataset_loader_svamp.load_gpt_preds(split='test')
    else:
        train_llm_rationales, train_llm_labels = load_pred_fn(split='train')
        test_llm_rationales, test_llm_labels = load_pred_fn(split='test')

    # Add LLM predictions to datasets
    datasets['train'] = datasets['train'].add_column('llm_label', train_llm_labels)
    datasets['test'] = datasets['test'].add_column('llm_label', test_llm_labels)
    datasets['train'] = datasets['train'].add_column('llm_rationale', train_llm_rationales)
    datasets['test'] = datasets['test'].add_column('llm_rationale', test_llm_rationales)

    # Handle validation set predictions if it exists
    if dataset_loader.has_valid:
        valid_llm_rationales, valid_llm_labels = load_pred_fn(split='valid')
        datasets['valid'] = datasets['valid'].add_column('llm_label', valid_llm_labels)
        datasets['valid'] = datasets['valid'].add_column('llm_rationale', valid_llm_rationales)

#### Subsample Training Data if specified
if args.subsample < 1.0:
    logger.info(f'Subsampling training data to {args.subsample * 100}%')
    datasets['train'] = datasets['train'].train_test_split(test_size=1.0 - args.subsample, seed=args.run)['train']
    logger.info(f'New training size: {len(datasets["train"])}')

#### Create Validation Split if necessary
if 'valid' not in datasets:
    logger.info('Creating validation split from training data (10%)')
    train_valid_datasets = datasets['train'].train_test_split(test_size=0.1, seed=0) # Use fixed seed for consistency
    datasets = DatasetDict({
        'train': train_valid_datasets['train'],
        'valid': train_valid_datasets['test'],
        'test': datasets['test'],
    })
    logger.info(f'New training size: {len(datasets["train"])}, Validation size: {len(datasets["valid"])}')

#### Select Label Type for Training ('gt' or 'llm')
if args.label_type == 'gt':
    logger.info('Using ground truth labels for training.')
    pass # Keep original 'label' column
elif args.label_type == 'llm' and args.llm is not None:
    logger.info('Using LLM-generated labels for training.')
    # Optional: Calculate LLM label accuracy against ground truth
    acc_fn = compute_equation_acc if args.dataset in ['svamp', 'asdiv'] else compute_text_acc
    try:
        train_label_acc = acc_fn(datasets['train']['llm_label'], datasets['train']['label'])
        test_label_acc = acc_fn(datasets['test']['llm_label'], datasets['test']['label'])
        logger.info(f'LLM Label Accuracy vs GT: Train={train_label_acc:.4f}, Test={test_label_acc:.4f}')
    except Exception as e:
        logger.warning(f'Could not compute LLM label accuracy: {e}')

    # Replace 'label' column with 'llm_label'
    datasets['train'] = datasets['train'].remove_columns('label')
    datasets['train'] = datasets['train'].rename_column('llm_label', 'label')
    # Keep 'llm_label' in test/valid sets for potential analysis, but training uses the renamed 'label'
else:
    raise ValueError(f'Invalid label_type ({args.label_type}) or llm ({args.llm}) combination.')

#### Select Rationale Source (only if LLM preds were loaded)
if args.llm is not None:
    logger.info('Using LLM-generated rationales.')
    if 'rationale' in datasets['train'].column_names:
        datasets = datasets.remove_columns('rationale') # Remove original rationale if it exists
    datasets = datasets.rename_column('llm_rationale', 'rationale')
    # Ensure 'rationale' column exists in all splits if llm is used
    if 'rationale' not in datasets['valid'].column_names:
         datasets['valid'] = datasets['valid'].add_column('rationale', datasets['valid']['llm_rationale'])
    if 'rationale' not in datasets['test'].column_names:
         datasets['test'] = datasets['test'].add_column('rationale', datasets['test']['llm_rationale'])
elif 'rationale' not in datasets['train'].column_names:
    logger.warning('LLM not specified and no ground truth rationale found. Rationale-based training/evaluation might fail.')
    # Add dummy rationale if needed downstream and llm is None
    # datasets = datasets.map(lambda example: {'rationale': ''}) 

#### Preprocess for NLI datasets (combine premise and hypothesis)
tokenizer_for_preprocessing = AutoTokenizer.from_pretrained(args.from_pretrained) # Need tokenizer temporarily
if 'nli' in args.dataset or args.dataset == 'anli1':
    logger.info('Preprocessing NLI dataset: combining premise and hypothesis.')
    def combine_nli_inputs(example):
        # Ensure keys exist before accessing
        premise = example.get('premise', '')
        hypothesis = example.get('hypothesis', '')
        return {'input': tokenizer_for_preprocessing.eos_token.join([premise, hypothesis])}

    datasets = datasets.map(
        combine_nli_inputs,
        remove_columns=['premise', 'hypothesis'],
        batched=False # Process example by example
    )
del tokenizer_for_preprocessing # Clean up temporary tokenizer

logger.info('Dataset preparation complete.')
print(datasets)

In [ ]:
# 5. Tokenize Data

logger.info(f'Loading tokenizer: {args.from_pretrained}')
tokenizer = AutoTokenizer.from_pretrained(args.from_pretrained)

# Define tokenization function based on model type
if args.model_type == 'task_prefix' and args.llm is not None:
    logger.info('Using task_prefix tokenization (requires rationale)')
    if 'rationale' not in datasets['train'].column_names:
        raise ValueError("Rationale column ('rationale') not found in training data, but required for task_prefix model type.")
    def tokenize_function(examples):
        # Tokenize main input with 'predict: ' prefix
        model_inputs = tokenizer(['predict: ' + text for text in examples['input']], max_length=args.max_input_length, truncation=True)
        # Tokenize input with 'explain: ' prefix for rationale generation
        expl_model_inputs = tokenizer(['explain: ' + text for text in examples['input']], max_length=args.max_input_length, truncation=True)
        model_inputs['expl_input_ids'] = expl_model_inputs['input_ids']
        model_inputs['expl_attention_mask'] = expl_model_inputs['attention_mask']

        # Tokenize labels (for prediction task)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(examples['label'], max_length=args.gen_max_len, truncation=True)
            # Tokenize rationales (for explanation task)
            aux_labels = tokenizer(examples['rationale'], max_length=args.gen_max_len, truncation=True) # Use gen_max_len for rationale target too

        model_inputs['labels'] = labels['input_ids']
        model_inputs['aux_labels'] = aux_labels['input_ids']
        return model_inputs

elif args.model_type == 'standard':
    logger.info('Using standard tokenization')
    def tokenize_function(examples):
        # Tokenize main input
        model_inputs = tokenizer(examples['input'], max_length=args.max_input_length, truncation=True)
        # Tokenize labels
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(examples['label'], max_length=args.gen_max_len, truncation=True)
        model_inputs['labels'] = labels['input_ids']
        # If llm rationales exist, tokenize them for potential use in compute_metrics_aux
        if 'rationale' in examples:
             with tokenizer.as_target_tokenizer():
                 aux_labels = tokenizer(examples['rationale'], max_length=args.gen_max_len, truncation=True)
             model_inputs['aux_labels'] = aux_labels['input_ids']
        return model_inputs

else:
    # Handle case where model_type is 'task_prefix' but llm is None (no rationale available)
    if args.model_type == 'task_prefix' and args.llm is None:
        logger.warning("Model type is 'task_prefix' but no LLM rationale source ('llm' is None). Falling back to standard tokenization.")
        args.model_type = 'standard' # Override model type
        # Rerun the standard tokenization definition
        logger.info('Using standard tokenization')
        def tokenize_function(examples):
            model_inputs = tokenizer(examples['input'], max_length=args.max_input_length, truncation=True)
            with tokenizer.as_target_tokenizer():
                labels = tokenizer(examples['label'], max_length=args.gen_max_len, truncation=True)
            model_inputs['labels'] = labels['input_ids']
            return model_inputs
    else:
        raise ValueError(f"Invalid model_type ({args.model_type}) or llm ({args.llm}) combination for tokenization.")

# Determine columns to remove after tokenization
remove_cols = ['input', 'label']
if args.llm is not None:
    remove_cols.extend(['llm_label', 'rationale']) # llm_label might have been renamed to label earlier
elif 'rationale' in datasets['train'].column_names: # If GT rationale exists but llm=None
     remove_cols.append('rationale')
if 'llm_label' in datasets['train'].column_names: # Ensure llm_label is removed if it still exists
    if 'llm_label' not in remove_cols:
        remove_cols.append('llm_label')
if 'llm_rationale' in datasets['train'].column_names: # Ensure llm_rationale is removed if it still exists
     if 'llm_rationale' not in remove_cols:
        remove_cols.append('llm_rationale')

# Ensure we only try to remove columns that actually exist
final_remove_cols = [col for col in remove_cols if col in datasets['train'].column_names]
logger.info(f'Columns to remove after tokenization: {final_remove_cols}')

# Apply tokenization
logger.info('Tokenizing datasets...')
tokenized_datasets = datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=final_remove_cols
)

logger.info('Tokenization complete.')
print(tokenized_datasets)

In [ ]:
# 6. Define Compute Metrics Function

logger.info('Setting up compute_metrics function.')

if args.model_type == 'standard':
    # Standard model only predicts labels, but we might have aux_labels (rationales) for analysis
    logger.info('Using compute_metrics_aux (evaluates labels, potentially logs rationales if available)')
    if args.dataset not in ['svamp', 'asdiv']:
        compute_metrics = compute_metrics_text_aux(tokenizer)
    else:
        compute_metrics = compute_metrics_equation_aux(tokenizer)
elif args.model_type == 'task_prefix':
    # Task prefix model predicts both labels and rationales
    logger.info('Using compute_metrics (evaluates both labels and rationales)')
    if args.dataset not in ['svamp', 'asdiv']:
        compute_metrics = compute_metrics_text(tokenizer)
    else:
        compute_metrics = compute_metrics_equation(tokenizer)
else:
    raise ValueError(f'Cannot select compute_metrics for model_type: {args.model_type}')

logger.info('Compute metrics function ready.')

In [ ]:
# 7. Run Training and Evaluation

logger.info(f"Starting training and evaluation...")

# Detect last checkpoint if resuming
last_checkpoint = None
if os.path.isdir(args.output_dir):
    last_checkpoint = get_last_checkpoint(args.output_dir)
    if last_checkpoint is not None:
        logger.info(f"Checkpoint detected, resuming training from {last_checkpoint}")

# Call the main training function
train_result = train_and_evaluate(
    args=args,
    run=args.run, # Pass the run argument
    tokenizer=tokenizer,
    tokenized_datasets=tokenized_datasets, # Pass the dictionary containing all splits
    compute_metrics=compute_metrics
    # last_checkpoint argument removed as it's not expected by the function
)

logger.info(f"Training and evaluation finished.")

# Optionally print or save final metrics
if train_result:
    metrics = train_result.metrics
    logger.info(f"Final Train Metrics: {metrics}")
    # You might want to save metrics to a file as well
    # import json
    # with open(os.path.join(args.output_dir, "train_results.json"), "w") as f:
    #     json.dump(metrics, f, indent=4)

# Final evaluation on the test set (if test_dataset was provided)
# The train_and_evaluate function already handles test set evaluation 
# if a test_dataset is passed and predict_with_generate=True.
# You can find test results in the output directory (e.g., test_results.json)
logger.info(f"Test results (if generated) are saved in: {args.output_dir}")